# **Task 1 — Train the Perceptron on the OR Gate**

## 01 Introduction

**What is PyTorch?**

PyTorch is an open-source deep learning framework developed by Facebook's AI Research lab (FAIR). It provides:

* A flexible and dynamic tensor computation library (like NumPy but with GPU support)
* An intuitive autograd system for automatic differentiation
* A powerful module system (torch.nn) for building neural networks

PyTorch is widely used in both research and industry due to its ease of use and strong support for dynamic computational graphs.

**What is this Lab About?**

In this lab, you will:

* Implement a Perceptron from scratch using raw PyTorch tensors
* Build a Multilayer Perceptron (MLP) using torch.nn.Module
* Train the Perceptron on the AND gate (a linearly separable problem)
* Train the MLP on the XOR gate (a non-linearly separable problem)

These tasks will help you understand:

* How simple neural networks learn through gradient descent
* Why deeper networks and nonlinear activations are necessary
* How PyTorch enables automatic differentiation and training workflows

**Why Are We Building This From Scratch?**

Building the Perceptron from scratch allows you to:
* Understand how weights and biases are initialized and updated
* Manually implement the forward pass and training loop
* See how PyTorch’s autograd system works behind the scenes

Once you understand the fundamentals, you will move to using torch.nn, which provides a high-level, modular way to build more complex networks like the MLP.

This hands-on approach builds intuition about how neural networks function — an essential foundation before working with large models and real-world data.

## 02 Perceptron

Here we will import the necessary PyTorch modules:

* torch: for tensor operations and autograd
* torch.nn: for building neural networks
* torch.optim: for optimization algorithms like SGD

These are core components in building and training neural networks in PyTorch.

But what are tensors? Tensors are **multi-dimensional arrays** that generalize scalars, vectors, and matrices to higher dimensions. They are fundamental data structures used to represent and manipulate data in machine learning models, particularly in deep learning.

In [35]:
import torch
import torch.nn as nn
import torch.optim as optim

We define the input-output pairs for the OR gate, which is linearly separable:
* Inputs: Two binary values
* Output: 1 only if a input is 1, otherwise -1
This dataset will be used to train a perceptron model from scratch.

In [36]:
X = torch.tensor([[0., 0.],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])
y = torch.tensor([-1., 1., 1., 1.])

We manually initialize the weights and bias of the Perceptron. Since we are not using torch.nn, we use raw torch.tensor with requires_grad= false to allow PyTorch to compute gradients for us during backpropagation.

In [37]:
w = torch.randn(2, requires_grad=False)
b = torch.randn(1, requires_grad=False)
'''
# If you want to set the values manually
w = torch.tensor([0., 0.], dtype=torch.float32, requires_grad=False)
b = torch.tensor([0.], dtype=torch.float32, requires_grad=False)
'''
# Set learning rate
lr = 0.1

In this block, we train the perceptron using the **Perceptron Learning Rule**, a simple and intuitive way to update the weights based on classification errors.

The key idea is:

- For each input sample, compute the predicted label using a **step function** that outputs `1` or `-1`.
- If the prediction is **correct**, do nothing.
- If the prediction is **incorrect**, update the weights and bias using:

$$
\mathbf{w} \leftarrow \mathbf{w} + \eta \cdot y \cdot \mathbf{x}
$$

$$
b \leftarrow b + \eta \cdot y
$$

Where:
- $ \eta $ is the learning rate
- $ y \in \{-1, 1\} $ is the true label
- $ \mathbf{x} $ is the input vector

This rule adjusts the decision boundary only when the model makes a mistake, helping the perceptron converge to a solution for **linearly separable problems** like the OR gate.

We repeat this for multiple epochs and track the number of misclassified samples in each epoch. When misclassifications become zero, the perceptron has converged.


In [38]:
for epoch in range(30):
    errors = 0
    for i in range(4):
        xi = X[i]
        yi = y[i]

        #Compute the Weighted Sum
        z = torch.dot(w, xi) + b
        #Apply the Activation Function (Step Function)
        prediction = 1.0 if z >= 0 else -1.0

        # Only update if misclassified
        if prediction != yi:
            w += lr * yi * xi
            b += lr * yi
            errors += 1

    print(f"Epoch {epoch+1}, Misclassified: {errors}")
    if errors == 0:
        break

Epoch 1, Misclassified: 1
Epoch 2, Misclassified: 1
Epoch 3, Misclassified: 1
Epoch 4, Misclassified: 0


Here we display the final weight and bias after all the updates and test the final perceptron model against our input.

In [39]:
# Final weights and bias
print("\nFinal weights:", w)
print("Final bias:", b)

# Test model
print("\nPredictions:")
for i in range(4):
    xi = X[i]
    z = torch.dot(w, xi) + b
    prediction = 1.0 if z >= 0 else -1.0
    print(f"Input: {xi.tolist()}, Output: {prediction}, Target: {y[i].item()}")


Final weights: tensor([1.1162, 0.7454])
Final bias: tensor([-0.6458])

Predictions:
Input: [0.0, 0.0], Output: -1.0, Target: -1.0
Input: [0.0, 1.0], Output: 1.0, Target: 1.0
Input: [1.0, 0.0], Output: 1.0, Target: 1.0
Input: [1.0, 1.0], Output: 1.0, Target: 1.0


However, perceptrons can not work on linearly non-separable data. XOR is such an example. Let's test our code on XOR and fail.

In [40]:
# XOR data
X = torch.tensor([[0., 0.],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])
y = torch.tensor([[-1.], [1.], [1.], [-1.]])

In [41]:

# If you want to set the values manually
w = torch.tensor([0., 0.], dtype=torch.float32, requires_grad=False)
b = torch.tensor([0.], dtype=torch.float32, requires_grad=False)

# Set learning rate
lr = 0.1

# Increase the number of epochs as much as you want
for epoch in range(2000):
    errors = 0
    for i in range(4):
        xi = X[i]
        yi = y[i]

        z = torch.dot(w, xi) + b
        prediction = 1.0 if z >= 0 else -1.0  # Step activation

        # Only update if misclassified
        if prediction != yi:
            w += lr * yi * xi
            b += lr * yi
            errors += 1

    print(f"Epoch {epoch+1}, Misclassified: {errors}")
    if errors == 0:
        break

Epoch 1, Misclassified: 3
Epoch 2, Misclassified: 3
Epoch 3, Misclassified: 4
Epoch 4, Misclassified: 4
Epoch 5, Misclassified: 4
Epoch 6, Misclassified: 4
Epoch 7, Misclassified: 4
Epoch 8, Misclassified: 4
Epoch 9, Misclassified: 4
Epoch 10, Misclassified: 4
Epoch 11, Misclassified: 4
Epoch 12, Misclassified: 4
Epoch 13, Misclassified: 4
Epoch 14, Misclassified: 4
Epoch 15, Misclassified: 4
Epoch 16, Misclassified: 4
Epoch 17, Misclassified: 4
Epoch 18, Misclassified: 4
Epoch 19, Misclassified: 4
Epoch 20, Misclassified: 4
Epoch 21, Misclassified: 4
Epoch 22, Misclassified: 4
Epoch 23, Misclassified: 4
Epoch 24, Misclassified: 4
Epoch 25, Misclassified: 4
Epoch 26, Misclassified: 4
Epoch 27, Misclassified: 4
Epoch 28, Misclassified: 4
Epoch 29, Misclassified: 4
Epoch 30, Misclassified: 4
Epoch 31, Misclassified: 4
Epoch 32, Misclassified: 4
Epoch 33, Misclassified: 4
Epoch 34, Misclassified: 4
Epoch 35, Misclassified: 4
Epoch 36, Misclassified: 4
Epoch 37, Misclassified: 4
Epoch 38, 

# **Task 2 — Replace Tanh with Sigmoid**

## 03 Multilayer Perceptron (MLP)

The XOR gate is a non-linearly separable problem, which a single-layer perceptron cannot solve. To learn it, we build an MLP with one hidden layer and use non-linear activation functions. We define a custom MLP class using PyTorch's nn.Module. The network has:
* 2 input nodes
* 1 hidden layer with 4 neurons
* 1 output node

We use Sigmoid activation to introduce non-linearity, which is essential for solving XOR.

In [42]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.hidden = nn.Linear(2, 4)  # Input layer to hidden layer
        self.output = nn.Linear(4, 1)  # Hidden layer to output
        self.activation = nn.Sigmoid()  # Sigmoid activation
        # self.activation = nn.Tanh() # Tanh activation


    def forward(self, x):
        x = self.activation(self.hidden(x))
        x = self.activation(self.output(x))
        return x

We instantiate the MLP model and set:

* Loss function: MSELoss (Mean Squared Error)
* Optimizer: SGD (Stochastic Gradient Descent)

These are standard components for training a supervised learning model.

In [43]:
model = MLP()
criterion = nn.MSELoss()  # Mean squared error loss
optimizer = optim.SGD(model.parameters(), lr=0.01)  # Stochastic gradient descent

This loop trains the MLP for 2000 epochs. Each iteration includes:

* Forward pass: model makes predictions
* Loss computation
* Backward pass: compute gradients via autograd
* Optimizer step: update weights

The model learns to approximate the XOR function through non-linear transformation.

# ** Task 3 Remove optimizer.zero_grad()**

In [44]:
# for epoch in range(20000):
#     optimizer.zero_grad()
#     outputs = model(X)
#     loss = criterion(outputs, y)
#     loss.backward()
#     optimizer.step()

#     if (epoch+1) % 200 == 0:
#         print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

for epoch in range(500):

    outputs = model(X)
    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    if (epoch+1)%50==0:
        print(epoch+1, loss.item())

50 1.000656247138977
100 1.0000057220458984
150 1.0
200 1.0
250 1.0
300 1.0
350 1.0
400 1.0
450 1.0
500 1.0


Finally, we use the trained model to make predictions on the XOR inputs and round the outputs to -1 or 1. A successful model should predict [-1, 1, 1, -1] as expected from XOR logic.

In [33]:
with torch.no_grad():
    predictions = model(X)
    print("\nPredictions (rounded):")
    print(predictions.round())


Predictions (rounded):
tensor([[0.],
        [0.],
        [0.],
        [0.]])


# **Task 4 **

In [45]:
# XNOR
X = torch.tensor([[0.,0.],
                  [0.,1.],
                  [1.,0.],
                  [1.,1.]])

y = torch.tensor([[1.],
                  [-1.],
                  [-1.],
                  [1.]])

In [46]:
# If you want to set the values manually
w = torch.tensor([0., 0.], dtype=torch.float32, requires_grad=False)
b = torch.tensor([0.], dtype=torch.float32, requires_grad=False)

# Set learning rate
lr = 0.1

# Increase the number of epochs as much as you want
for epoch in range(2000):
    errors = 0
    for i in range(4):
        xi = X[i]
        yi = y[i]

        z = torch.dot(w, xi) + b
        prediction = 1.0 if z >= 0 else -1.0  # Step activation

        # Only update if misclassified
        if prediction != yi:
            w += lr * yi * xi
            b += lr * yi
            errors += 1

    print(f"Epoch {epoch+1}, Misclassified: {errors}")
    if errors == 0:
        break

Epoch 1, Misclassified: 2
Epoch 2, Misclassified: 3
Epoch 3, Misclassified: 4
Epoch 4, Misclassified: 4
Epoch 5, Misclassified: 4
Epoch 6, Misclassified: 4
Epoch 7, Misclassified: 4
Epoch 8, Misclassified: 4
Epoch 9, Misclassified: 4
Epoch 10, Misclassified: 4
Epoch 11, Misclassified: 4
Epoch 12, Misclassified: 4
Epoch 13, Misclassified: 4
Epoch 14, Misclassified: 4
Epoch 15, Misclassified: 4
Epoch 16, Misclassified: 4
Epoch 17, Misclassified: 4
Epoch 18, Misclassified: 4
Epoch 19, Misclassified: 4
Epoch 20, Misclassified: 4
Epoch 21, Misclassified: 4
Epoch 22, Misclassified: 4
Epoch 23, Misclassified: 4
Epoch 24, Misclassified: 4
Epoch 25, Misclassified: 4
Epoch 26, Misclassified: 4
Epoch 27, Misclassified: 4
Epoch 28, Misclassified: 4
Epoch 29, Misclassified: 4
Epoch 30, Misclassified: 4
Epoch 31, Misclassified: 4
Epoch 32, Misclassified: 4
Epoch 33, Misclassified: 4
Epoch 34, Misclassified: 4
Epoch 35, Misclassified: 4
Epoch 36, Misclassified: 4
Epoch 37, Misclassified: 4
Epoch 38, 

In [47]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.hidden = nn.Linear(2, 4)  # Input layer to hidden layer
        self.output = nn.Linear(4, 1)  # Hidden layer to output
        # self.activation = nn.Sigmoid()  # Sigmoid activation
        self.activation = nn.Tanh() # Tanh activation


    def forward(self, x):
        x = self.activation(self.hidden(x))
        x = self.activation(self.output(x))
        return x

In [48]:
model = MLP()
criterion = nn.MSELoss()  # Mean squared error loss
optimizer = optim.SGD(model.parameters(), lr=0.01)  # Stochastic gradient descent

In [49]:
# for epoch in range(20000):
#     optimizer.zero_grad()
#     outputs = model(X)
#     loss = criterion(outputs, y)
#     loss.backward()
#     optimizer.step()

#     if (epoch+1) % 200 == 0:
#         print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

for epoch in range(500):

    outputs = model(X)
    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    if (epoch+1)%50==0:
        print(epoch+1, loss.item())

50 0.8959676027297974
100 0.02795821987092495
150 0.0003540871839504689
200 1.190302282338962e-05
250 0.00013370240048971027
300 6.246398953635435e-10
350 7.52118367586263e-11
400 9.060308059360977e-12
450 9.094947017729282e-13
500 8.881784197001252e-14


In [50]:
with torch.no_grad():
    predictions = model(X)
    print("\nPredictions (rounded):")
    print(predictions.round())


Predictions (rounded):
tensor([[ 1.],
        [-1.],
        [-1.],
        [ 1.]])


The same MLP architecture also solves XNOR because XNOR is a nonlinear problem similar to XOR. The hidden layer enables the network to learn nonlinear decision boundaries needed for correct classification.

# **Task 5**

**1.Why can a single perceptron not solve XOR?**

A single perceptron cannot solve XOR because XOR is not linearly separable.

**2.Why does the MLP succeed where the perceptron fails?**

An MLP succeeds because its hidden layer and nonlinear activation functions can learn nonlinear decision boundaries.